# Segment scorecard evaluation

Q1: is the pooled logistic score good on each business segment?
Q2: is a same-predictor WoE refit worth a split, or is intercept/slope recalibration enough?

This notebook uses a synthetic book. The data-setup cell shows the interactive confirmation flow (metadata, capability matrix, settings cache).

In [ ]:
from scorecard_segment_eval import (
    Gates,
    action_list,
    build_final_report,
    confirm_and_save_plan,
    decision_table,
    evaluate_segments,
    load_scorecard_table,
    make_synthetic_book,
    write_final_report,
)

df, cols = make_synthetic_book(n=12000, seed=7)
df.head()

## Confirm the analysis plan

`load_scorecard_table` only requires `col_id`, `col_score`, and `cols_pred_used`. Optional fields are inferred or joined via `sql/enrich_by_id.sql`. Review the capability matrix, then save a settings cache (set `auto_confirm=False` in a live notebook to click through ipywidgets / `input()`).

In [ ]:
loaded, spec, plan = load_scorecard_table(
    df,
    col_id="SKP_CREDIT_CASE",
    col_score="pd",
    cols_pred_used=["x1_woe", "x2", "cat"],
    col_date="score_date",
    col_target="default",
    col_obs="obs",
    cols_segment=["channel"],
    cols_pred=["x1", "x2", "cat"],
    cols_pred_woe=["x1_woe"],
    grouping_path="grouping.json",
    warn=True,
    auto_confirm=True,
    cache_filename="analysis_plan.json",
)
print(plan.capabilities)
spec.to_columns()

In [ ]:
gates = Gates(n_jobs=2, woe_strategy="tree", submodel=False)
result = evaluate_segments(df, cols, gates)
decision_table(result)

Important + WEAK segments are the Q2 action list. `SPLIT` requires holdout ΔGini ≥ 0.03, a positive CI, Brier or log-loss improvement, and WoE-shape divergence.

In [ ]:
action_list(result)

In [ ]:
result.refit_comparison

In [ ]:
result.characteristics.sort_values("psi", ascending=False).head(12)

AR-aligned Gini is filled when a segment's approval rate is far from the lower-AR sibling / overall rate. The report writes HTML + Markdown recommendations (split vs recalibrate vs keep pooled, plus stability notes).

In [ ]:
result.decisions[["segment_value", "approval_rate", "gini", "gini_ar_aligned", "q2_action"]]
html_path, md_path = write_final_report(result, "segment_eval_report.html")
html_path, md_path